# KLDM Data Pipeline Tutorial

This notebook explains the full data pipeline in this repository as a step-by-step lecture.

We go in the same order a new reader should think about the system:

1. Start from the raw MP-20 CSV files
2. Inspect what one CIF entry looks like after parsing
3. Convert raw CSV rows into tensor-backed `torch_geometric.data.Data` objects
4. Define the base dataset abstraction used everywhere
5. Specialize that abstraction for the two tasks:
   - CSP
   - DNG / de-novo generation
6. Check the final batch contract against what KLDM expects in `forward`, `training_targets`, and sampling

Throughout the notebook, the important target is the **KLDM-ready batch** with these fields:

- `pos`: node positions
- `h`: atomic species
- `l`: graph-level lattice vector of shape `[num_graphs, 6]`
- `batch`: graph assignment for each node
- `edge_node_index`: node graph connectivity

That is the minimum contract needed by the KLDM code path shown in the paper implementation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Batch, Data

from kldm.data import DatasetCSP, DatasetDNG
from kldm.data.convertToTensor import load_mp20_split, preprocess_csv, process_cif

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data' / 'mp_20'
TRAIN_CSV = DATA_DIR / 'train.csv'
VAL_CSV = DATA_DIR / 'val.csv'
TEST_CSV = DATA_DIR / 'test.csv'
TRAIN_PT = DATA_DIR / 'train.pt'

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(0)
np.random.seed(0)


## 1. Raw MP-20 data

The raw source data lives in CSV files. Each row contains a CIF string representing a crystal.

The first thing to understand is that the repository does **not** start from graphs. It starts from:

- a row in a CSV file
- a CIF string in the `cif` column
- parsing that CIF into crystal structure information

So before thinking about KLDM, we should look at the raw files directly.


In [ ]:
csv_paths = [TRAIN_CSV, VAL_CSV, TEST_CSV]
summary = pd.DataFrame(
    {
        'split': [p.stem for p in csv_paths],
        'exists': [p.exists() for p in csv_paths],
        'size_mb': [round(p.stat().st_size / (1024**2), 2) if p.exists() else None for p in csv_paths],
    }
)
summary


In [ ]:
train_df = pd.read_csv(TRAIN_CSV, nrows=5)
print('Columns:', list(train_df.columns))
train_df.head(2)


In [ ]:
first_cif = train_df.loc[0, 'cif']
print(first_cif[:1200] + ('...' if len(first_cif) > 1200 else ''))


### What `process_cif(...)` does

`process_cif(...)` is the first true conversion step.

It turns a raw CIF string into a PyG `Data` object with the minimal crystal fields:

- `pos`: fractional coordinates
- `h`: atomic numbers
- `lengths`: lattice lengths `(a, b, c)`
- `angles`: lattice angles `(alpha, beta, gamma)`

This is the clean crystal representation before the dataset abstraction adds KLDM-ready fields like `l` and `edge_node_index`.


In [ ]:
parsed = process_cif(first_cif)
print(parsed)
print('pos shape   :', tuple(parsed.pos.shape))
print('h shape     :', tuple(parsed.h.shape))
print('lengths     :', parsed.lengths)
print('angles      :', parsed.angles)


In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
pos = parsed.pos.cpu().numpy()
h = parsed.h.cpu().numpy()
scatter = ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=h, cmap='viridis', s=50)
ax.set_title('One parsed MP-20 crystal from raw CIF')
ax.set_xlabel('x (fractional)')
ax.set_ylabel('y (fractional)')
ax.set_zlabel('z (fractional)')
fig.colorbar(scatter, ax=ax, label='atomic number')
plt.show()


## 2. From CSV to tensor files

The next stage is `convertToTensor.py`.

Its job is simple:

1. read each split CSV
2. parse every CIF row
3. save the split as a `.pt` file containing `Data` objects
4. save simple lattice statistics per atom count

This is where the raw dataset becomes something that can be loaded efficiently during training.


In [ ]:
pt_summary = pd.DataFrame(
    {
        'split': ['train', 'val', 'test'],
        'pt_exists': [(DATA_DIR / f'{split}.pt').exists() for split in ['train', 'val', 'test']],
        'pt_size_mb': [round((DATA_DIR / f'{split}.pt').stat().st_size / (1024**2), 2) if (DATA_DIR / f'{split}.pt').exists() else None for split in ['train', 'val', 'test']],
    }
)
pt_summary


In [ ]:
train_split = load_mp20_split(DATA_DIR, 'train')
print('Number of tensor samples in train split:', len(train_split))
print('Type of one element:', type(train_split[0]))
print(train_split[0])


### Distribution of the real MP-20 data

Before we define any dataset classes, it is useful to understand the structure of the real training data statistically.


In [ ]:
num_atoms = np.array([sample.pos.shape[0] for sample in train_split])
lengths = np.stack([sample.lengths.view(-1).numpy() for sample in train_split])
angles = np.stack([sample.angles.view(-1).numpy() for sample in train_split])

stats_df = pd.DataFrame(
    {
        'metric': ['num_structures', 'mean_atoms', 'min_atoms', 'max_atoms', 'mean_a', 'mean_b', 'mean_c'],
        'value': [
            len(train_split),
            float(num_atoms.mean()),
            int(num_atoms.min()),
            int(num_atoms.max()),
            float(lengths[:, 0].mean()),
            float(lengths[:, 1].mean()),
            float(lengths[:, 2].mean()),
        ],
    }
)
stats_df


In [ ]:
fig = plt.figure(figsize=(14, 10))
ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 2)
ax3 = fig.add_subplot(2, 2, 3)
ax4 = fig.add_subplot(2, 2, 4, projection='3d')

ax1.hist(num_atoms, bins=30, color='steelblue', edgecolor='black')
ax1.set_title('MP-20: atoms per structure')
ax1.set_xlabel('number of atoms')

ax2.hist(lengths[:, 0], bins=30, alpha=0.7, label='a')
ax2.hist(lengths[:, 1], bins=30, alpha=0.7, label='b')
ax2.hist(lengths[:, 2], bins=30, alpha=0.7, label='c')
ax2.set_title('MP-20: lattice lengths')
ax2.set_xlabel('length')
ax2.legend()

ax3.hist(angles[:, 0], bins=30, alpha=0.7, label='alpha')
ax3.hist(angles[:, 1], bins=30, alpha=0.7, label='beta')
ax3.hist(angles[:, 2], bins=30, alpha=0.7, label='gamma')
ax3.set_title('MP-20: lattice angles')
ax3.set_xlabel('degrees')
ax3.legend()

sample = train_split[0]
ax4.scatter(sample.pos[:, 0], sample.pos[:, 1], sample.pos[:, 2], c=sample.h, cmap='plasma', s=40)
ax4.set_title('One real MP-20 crystal')
ax4.set_xlabel('x')
ax4.set_ylabel('y')
ax4.set_zlabel('z')

plt.tight_layout()
plt.show()


## 3. The base dataset abstraction

Now we move from raw tensor files to the **base dataset class** used by the rest of the pipeline.

The key design choice in this repository is:

- raw conversion stores `pos`, `h`, `lengths`, `angles`
- the dataset layer adds the fields that KLDM actually wants at training time

So the dataset layer is where a generic crystal sample becomes a **KLDM-ready graph sample**.

For each sample, the base dataset prepares:

- `edge_node_index`: fully connected directed graph
- `l`: lattice vector `[a, b, c, alpha, beta, gamma]`

This means downstream code can use the batch directly in KLDM-style `forward(...)` and `training_targets(...)`.


In [ ]:
mp20_dataset = DatasetDNG(path=TRAIN_PT)
mp20_sample = mp20_dataset[0]
print(mp20_sample)
print('pos shape          :', tuple(mp20_sample.pos.shape))
print('h shape            :', tuple(mp20_sample.h.shape))
print('l shape            :', tuple(mp20_sample.l.shape))
print('edge_node_index    :', tuple(mp20_sample.edge_node_index.shape))
print('has lengths/angles :', hasattr(mp20_sample, 'lengths'), hasattr(mp20_sample, 'angles'))


In [ ]:
mp20_batch = Batch.from_data_list([mp20_dataset[0], mp20_dataset[1], mp20_dataset[2]])
print(mp20_batch)
print('batch.pos            :', tuple(mp20_batch.pos.shape))
print('batch.h              :', tuple(mp20_batch.h.shape))
print('batch.l              :', tuple(mp20_batch.l.shape))
print('batch.batch          :', tuple(mp20_batch.batch.shape))
print('batch.edge_node_index:', tuple(mp20_batch.edge_node_index.shape))


### Why this matters for KLDM

At this point, a batch already fits the central interface expected by KLDM:

- node-level tensors: `pos`, `h`
- graph-level tensor: `l`
- graph assignment: `batch`
- connectivity: `edge_node_index`

That is exactly why the dataset layer exists in this form.


In [ ]:
def summarize_kldm_batch(batch: Batch | Data, name: str) -> None:
    num_graphs = batch.num_graphs if isinstance(batch, Batch) else 1
    node_index = batch.batch if getattr(batch, 'batch', None) is not None else torch.zeros(batch.pos.shape[0], dtype=torch.long)
    t_graph = torch.rand(num_graphs)
    t_nodes = t_graph[node_index]
    v0 = torch.zeros_like(batch.pos)

    print(name)
    print('  pos              :', tuple(batch.pos.shape))
    print('  v0               :', tuple(v0.shape))
    print('  h                :', tuple(batch.h.shape))
    print('  l                :', tuple(batch.l.shape))
    print('  batch            :', tuple(node_index.shape))
    print('  edge_node_index  :', tuple(batch.edge_node_index.shape))
    print('  graph timestep   :', tuple(t_graph.shape))
    print('  node timestep    :', tuple(t_nodes.shape))

summarize_kldm_batch(mp20_batch, 'KLDM-ready MP-20 batch')


## 4. Task-specific dataset: CSP

Now we specialize the base dataset abstraction.

For **CSP**, the starting point is not a full observed crystal. Instead, we start from a **chemical formula** and create a graph with:

- atom identities `h` determined by the formula
- placeholder coordinates `pos`
- default lattice vector `l`
- fully connected graph edges

So CSP is a **composition-conditioned generation** setup.


In [ ]:
csp_dataset = DatasetCSP(
    formulas=['SiO2', 'LiFePO4', 'NaCl'],
    n_samples_per_formula=3,
)

csp_sample = csp_dataset[0]
print(csp_sample)
print('pos shape       :', tuple(csp_sample.pos.shape))
print('h shape         :', tuple(csp_sample.h.shape))
print('l shape         :', tuple(csp_sample.l.shape))
print('edge_node_index :', tuple(csp_sample.edge_node_index.shape))


In [ ]:
csp_batch = Batch.from_data_list([csp_dataset[0], csp_dataset[1], csp_dataset[2], csp_dataset[3]])
summarize_kldm_batch(csp_batch, 'KLDM-ready CSP batch')


In [ ]:
csp_counts = torch.bincount(csp_batch.batch).cpu().numpy()
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

ax1.bar(np.arange(len(csp_counts)), csp_counts, color='darkorange')
ax1.set_title('CSP: atoms per graph in one batch')
ax1.set_xlabel('graph index')
ax1.set_ylabel('number of atoms')

mask = csp_batch.batch == 0
pos0 = csp_batch.pos[mask].cpu().numpy()
h0 = csp_batch.h[mask].cpu().numpy()
scatter = ax2.scatter(pos0[:, 0], pos0[:, 1], pos0[:, 2], c=h0, cmap='viridis', s=60)
ax2.set_title('One CSP graph')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')
fig.colorbar(scatter, ax=ax2, label='atomic number')
plt.tight_layout()
plt.show()


## 5. Task-specific dataset: DNG / de-novo generation

For **DNG**, this repository now uses the real MP-20 crystal splits directly.

That means the de-novo path is:

1. load a real crystal from MP-20
2. keep its atomic species `h`
3. keep its crystal coordinates `pos`
4. collapse lattice information into the graph-level `l` vector
5. attach the fully connected node graph

So DNG is now only the real KLDM-relevant path, without extra synthetic loader logic.


In [ ]:
dng_dataset = DatasetDNG(path=TRAIN_PT)

dng_sample = dng_dataset[0]
print(dng_sample)
print('pos shape       :', tuple(dng_sample.pos.shape))
print('h shape         :', tuple(dng_sample.h.shape))
print('l shape         :', tuple(dng_sample.l.shape))
print('edge_node_index :', tuple(dng_sample.edge_node_index.shape))


In [ ]:
dng_batch = Batch.from_data_list([dng_dataset[0], dng_dataset[1], dng_dataset[2], dng_dataset[3]])
summarize_kldm_batch(dng_batch, 'KLDM-ready DNG batch')


In [ ]:
dng_counts = [dng_dataset[i].pos.shape[0] for i in range(len(dng_dataset))]
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

ax1.hist(dng_counts, bins=np.arange(min(dng_counts), max(dng_counts) + 2) - 0.5, color='seagreen', edgecolor='black')
ax1.set_title('DNG: atoms per crystal')
ax1.set_xlabel('number of atoms')

mask = dng_batch.batch == 0
pos0 = dng_batch.pos[mask].cpu().numpy()
h0 = dng_batch.h[mask].cpu().numpy()
scatter = ax2.scatter(pos0[:, 0], pos0[:, 1], pos0[:, 2], c=h0, cmap='cividis', s=60)
ax2.set_title('One DNG graph')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')
fig.colorbar(scatter, ax=ax2, label='atomic number')
plt.tight_layout()
plt.show()


## 6. Final checklist: does the pipeline satisfy the KLDM input contract?

We finish by checking all three dataset paths against the fields that KLDM uses.

This is the practical summary a new reader should remember:

- `convertToTensor.py` converts raw CIF strings into crystal tensors
- `Dataset` loads real MP-20 crystal samples
- `DatasetCSP` creates formula-conditioned graphs
- `DatasetDNG` creates synthetic de-novo graphs
- `KLDMDataset` is the layer that guarantees a KLDM-ready sample shape

So if a model expects `pos`, `h`, `l`, `batch`, and `edge_node_index`, these datasets are the entry point that provide them.


In [ ]:
def checklist_row(name: str, batch: Batch | Data) -> dict:
    return {
        'dataset': name,
        'has_pos': hasattr(batch, 'pos'),
        'has_h': hasattr(batch, 'h'),
        'has_l': hasattr(batch, 'l'),
        'has_batch_or_single': isinstance(batch, Data) or hasattr(batch, 'batch'),
        'has_edge_node_index': hasattr(batch, 'edge_node_index'),
        'pos_shape': tuple(batch.pos.shape),
        'h_shape': tuple(batch.h.shape),
        'l_shape': tuple(batch.l.shape),
    }

pd.DataFrame([
    checklist_row('MP-20 batch', mp20_batch),
    checklist_row('CSP batch', csp_batch),
    checklist_row('DNG batch', dng_batch),
])
